# Part 11: 디버깅 & 최적화**소요 시간**: 약 1.5시간**난이도**: ⭐⭐⭐⭐ (중상급)**목표**: GraphRAG 실패 패턴을 분류하고, LangSmith 추적 + 프롬프트/인프라 최적화를 적용한다.---## 학습 순서1. 환경 설정2. 실패 패턴 분류학 (5가지 실패 유형)3. LangSmith 추적 파이프라인4. 프롬프트 최적화 (Few-shot, Chain-of-Thought)5. 인프라 최적화 (캐싱, 인덱스, 배치)6. 종합 트러블슈팅 워크숍7. 연습 문제

---## 1. 환경 설정### 패키지 설치 및 Neo4j 연결

In [ ]:
import os, json, timefrom dotenv import load_dotenvfrom neo4j import GraphDatabaseload_dotenv()load_dotenv(dotenv_path="../.env")NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))try:    driver.verify_connectivity()    print("[OK] Neo4j 연결 성공!")except Exception as e:    print(f"[ERROR] 연결 실패: {e}")def run_query(query, parameters=None, print_result=True):    with driver.session() as session:        result = session.run(query, parameters or {})        records = [record.data() for record in result]        if print_result:            for i, rec in enumerate(records, 1):                print(f"  [{i}] {rec}")            if not records:                print("  (결과 없음)")        return recordsprint("[OK] 환경 설정 완료")

---## 2. 실패 패턴 분류학GraphRAG 파이프라인의 5가지 대표 실패 유형:| # | 실패 유형 | 증상 | 해결 ||---|----------|------|------|| 1 | **스키마 불일치** | "Property not found" | 온톨로지 재검토 || 2 | **Cypher 문법 오류** | "SyntaxError" | Few-shot 프롬프트 || 3 | **빈 결과** | 답변이 "모르겠습니다" | 검색 범위 확대 || 4 | **환각** | KG에 없는 엔티티 언급 | Entity 검증 || 5 | **성능 저하** | 응답 > 5초 | 인덱스/캐싱 |> **핵심**: 80%의 실패는 1-3번(검색 실패)에서 발생. 검색부터 디버깅하라.

In [ ]:
# 실패 패턴 진단 함수def diagnose_failure(query: str, cypher: str, result: list, answer: str) -> dict:    """GraphRAG 실패 패턴을 진단합니다."""    diagnosis = {"query": query, "issues": []}    # 1. Cypher 문법 오류 체크    if isinstance(result, dict) and "error" in result:        diagnosis["issues"].append({            "type": "CYPHER_SYNTAX_ERROR",            "severity": "HIGH",            "fix": "Few-shot 프롬프트에 올바른 Cypher 예시 추가"        })    # 2. 빈 결과 체크    elif not result:        diagnosis["issues"].append({            "type": "EMPTY_RESULT",            "severity": "MEDIUM",            "fix": "검색 범위 확대 (depth 증가, 관계 타입 추가)"        })    # 3. 답변 품질 체크    elif "모르겠" in answer or "없습니다" in answer:        diagnosis["issues"].append({            "type": "LOW_QUALITY_ANSWER",            "severity": "MEDIUM",            "fix": "Context 보강 또는 프롬프트 개선"        })    return diagnosisprint("[OK] 진단 함수 준비 완료")

---## 3. LangSmith 추적 파이프라인LangSmith를 사용하면 LLM 호출의 전 과정을 추적할 수 있습니다.### 셋업```pythonimport osos.environ["LANGCHAIN_TRACING_V2"] = "true"os.environ["LANGCHAIN_API_KEY"] = "ls-..."os.environ["LANGCHAIN_PROJECT"] = "graphrag-debug"```### 추적 대상| 구간 | 측정 항목 ||------|----------|| Cypher 생성 | 프롬프트 → 생성된 Cypher || Neo4j 실행 | 쿼리 시간, 결과 수 || 답변 생성 | 컨텍스트 크기, 토큰 수 || 전체 | End-to-end 지연시간, 비용 |

In [ ]:
# 간단한 추적 데코레이터 (LangSmith 없이도 사용 가능)import functoolsquery_log = []def trace(func):    @functools.wraps(func)    def wrapper(*args, **kwargs):        start = time.time()        try:            result = func(*args, **kwargs)            duration = time.time() - start            query_log.append({                "function": func.__name__,                "duration_s": round(duration, 3),                "status": "success",                "timestamp": time.strftime("%H:%M:%S")            })            return result        except Exception as e:            duration = time.time() - start            query_log.append({                "function": func.__name__,                "duration_s": round(duration, 3),                "status": "error",                "error": str(e)            })            raise    return wrapper@tracedef traced_query(cypher):    return run_query(cypher, print_result=False)# 테스트traced_query("MATCH (n) RETURN count(n) AS cnt")print(f"추적 로그: {query_log}")

---## 4. 프롬프트 최적화### Few-shot 프롬프트Text2Cypher의 정확도를 높이는 가장 효과적인 방법:```시스템 프롬프트:  스키마: (Company)-[:INVESTS_IN]->(Company)  예시 1:    질문: "삼성전자에 투자한 기관은?"    Cypher: MATCH (i)-[:INVESTS_IN]->(c:Company {name: '삼성전자'}) RETURN i  예시 2:    질문: "국민연금이 투자한 기업이 개발한 제품은?"    Cypher: MATCH (i {name: '국민연금'})-[:INVESTS_IN]->(c)-[:DEVELOPS]->(p) RETURN p사용자 질문: {user_question}```

In [ ]:
# Few-shot 프롬프트 템플릿FEW_SHOT_TEMPLATE = '''당신은 Neo4j Cypher 쿼리 생성 전문가입니다.### 스키마{schema}### 예시질문: "삼성전자에 투자한 기관은?"Cypher: MATCH (i)-[:INVESTS_IN]->(c:Company {{name: '삼성전자'}}) RETURN i.name질문: "국민연금이 투자한 기업이 개발한 제품은?"Cypher: MATCH (i {{name: '국민연금'}})-[:INVESTS_IN]->(c)-[:DEVELOPS]->(p) RETURN c.name, p.name### 규칙1. 항상 RETURN 절에 .name 속성을 포함하세요2. 문자열 비교 시 정확한 이름을 사용하세요3. 관계 방향에 주의하세요### 질문{question}### Cypher'''print("=== Few-shot 프롬프트 템플릿 ===")print(FEW_SHOT_TEMPLATE[:300] + "...")

---## 5. 인프라 최적화### 7가지 최적화 기법| # | 기법 | 효과 | 난이도 ||---|------|------|--------|| 1 | 인덱스 추가 | 쿼리 10배 빠름 | 쉬움 || 2 | 쿼리 캐싱 | 반복 쿼리 즉시 응답 | 쉬움 || 3 | Connection Pool | 연결 오버헤드 감소 | 보통 || 4 | 배치 UNWIND | 대량 적재 10배 빠름 | 보통 || 5 | APOC 프로시저 | 복잡 쿼리 최적화 | 보통 || 6 | 메모리 튜닝 | 전체 성능 향상 | 어려움 || 7 | Read Replica | 읽기 확장성 | 어려움 |

In [ ]:
# 최적화 1: 인덱스 추가print("=== 인덱스 추가 ===")index_queries = [    "CREATE INDEX IF NOT EXISTS FOR (c:Company) ON (c.name)",    "CREATE INDEX IF NOT EXISTS FOR (p:Person) ON (p.name)",    "CREATE INDEX IF NOT EXISTS FOR (e:Equipment) ON (e.name)",]for q in index_queries:    try:        run_query(q, print_result=False)        print(f"  [OK] {q.split('FOR')[1].strip()}")    except Exception as e:        print(f"  [SKIP] {e}")# 최적화 2: 쿼리 캐싱from functools import lru_cache@lru_cache(maxsize=128)def cached_query(cypher: str) -> str:    result = run_query(cypher, print_result=False)    return json.dumps(result, ensure_ascii=False)print("\n[OK] LRU 캐시 (128개) 설정 완료")

---## 6. 연습 문제### 문제 1: 실패 쿼리 10개 디버깅의도적으로 잘못된 Cypher 10개를 준비하고, 각각 어떤 실패 패턴인지 진단하세요.### 문제 2: EXPLAIN/PROFILE 분석`PROFILE MATCH (c:Company {name: '삼성전자'})-[*1..3]-(n) RETURN n` 실행 후 실행 계획을 분석하세요.### 문제 3: 캐시 히트율 측정100개 쿼리를 실행하면서 캐시 히트율을 측정하세요.

In [ ]:
# [연습] 여기에 코드를 작성하세요print("디버깅 & 최적화 연습 문제를 풀어보세요!")

---## 7. 정리| 영역 | 핵심 기법 ||------|----------|| 실패 진단 | 5가지 패턴 분류 || 추적 | LangSmith 또는 커스텀 데코레이터 || 프롬프트 | Few-shot + Chain-of-Thought || 인프라 | 인덱스, 캐싱, Connection Pool |> **"디버깅의 80%는 검색 실패에서 시작된다. 검색부터 고쳐라."**### 다음 Part 12: 엔터프라이즈 GraphRAG

In [ ]:
# 세션 정리# driver.close()# print("[OK] Neo4j 드라이버 종료 완료")print("실습을 마칩니다. 수고하셨습니다!")